In [ ]:
# Cell 1: Setup and imports
%env XLA_PYTHON_CLIENT_PREALLOCATE=False
import jax
jax.config.update("jax_enable_x64", True)
import jax.numpy as jnp
import pickle
import os
import matplotlib.pyplot as plt
import numpy as np

DIRS = ["./power2WF/heavyfactor4","./power2WF/heavyfactor05", "./power2/heavyfactor4", "./power2/heavyfactor05"]
data = {}  # keyed by dim
for DIR in DIRS:
    data[DIR] = {}
    _data = {}
    for file in os.listdir(DIR):
        if file.endswith(".pkl") and file.startswith("_median"):
            with open(os.path.join(DIR, file), "rb") as f:
                my_pkl = pickle.load(f)
    
            dim = my_pkl['config']['dim']
            true_Z = float(jnp.exp(my_pkl['config']['logZ']))
            result = my_pkl['res'][4]
            if dim in _data.keys():
                _data[dim]['logZoverZ'] = jnp.concatenate([ _data[dim]['logZoverZ'] , result], axis=0)
            else:
                _data[dim] = {
                'logZoverZ': result,
                'true_Z': true_Z
                }
    eps = 0.001
    J = 10
    for dim in _data.keys():
        n, T = _data[dim]["logZoverZ"].shape
        max_n = n//J * J
        logZoverZ = _data[dim]["logZoverZ"][:max_n]
        logZoverZ = logZoverZ.reshape((max_n//J, J, T))
    #    ZoverZ = jnp.exp(logZoverZ)
        Zmed = jnp.prod(jnp.exp(jnp.median(logZoverZ, axis=1)), axis=-1)
        _data[dim]["Zmed"] = Zmed
        # Median over the J runs of the products of the ratio estimates,
        # i.e. median of the J independent estimates \hat{Z}_T^{(j)}
        #Zmed_prime = jnp.exp(jnp.median(logZoverZ.sum(axis=-1), axis=1))
        #_data[dim]["Zmed_prime"] = Zmed_prime
    data[DIR] = _data.copy()

In [ ]:
data2 = {}
for DIR in DIRS:
    data2[DIR] = {}
    _data2 = {}
    for file in os.listdir(DIR):
        if file.endswith(".pkl") and file.startswith("__"):
            with open(os.path.join(DIR, file), "rb") as f:
                my_pkl = pickle.load(f)
    
            dim = my_pkl['config']['dim']
            true_Z = float(jnp.exp(my_pkl['config']['logZ']))
            result = my_pkl['res'][4]
            if dim in _data2.keys():
                _data2[dim]['logZoverZ'] = jnp.concatenate([_data2[dim]['logZoverZ'] , result], axis=0)
            else:
                _data2[dim] = {
                'logZoverZ': result,
                'true_Z': true_Z
                }
    for dim in _data2.keys():
        _data2[dim]["Z"] = jnp.exp(_data2[dim]['logZoverZ'].sum(axis=-1))
    data2[DIR] = _data2.copy()

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(10, 10), sharey=True)
axes = axes.flatten()

titles = {"./power2WF/heavyfactor4":"Heavy-tailed","./power2WF/heavyfactor05":"Light-tailed","./power2/heavyfactor4":"Heavy-tailed","./power2/heavyfactor05":"Light-tailed"}

for ax_idx, (DIR, ax) in enumerate(zip(DIRS, axes)):
    dims = list(data[DIR].keys())
    dims.sort()
    dims = list(filter(lambda dim: dim != 100, dims))

    box1 = [np.abs(data[DIR][dim]["Zmed"][:200] / data[DIR][dim]["true_Z"]-1) for dim in dims]
    box2 = [np.abs(data2[DIR][dim]["Z"][:200]   / data[DIR][dim]["true_Z"]-1) for dim in dims]
    #box3 = [np.abs(data[DIR][dim]["Zmed_prime"][:200] / data[DIR][dim]["true_Z"]-1) for dim in dims]


    #COMMENTED OUT FOR MEDIAN OF Z
    #positions1 = [i - 0.26 for i in range(len(dims))]
    #positions3 = [i for i in range(len(dims))]
    #positions2 = [i + 0.26 for i in range(len(dims))]


    positions1 = [i - 0.18 for i in range(len(dims))]
    positions2 = [i + 0.18 for i in range(len(dims))]

    common_props = dict(linewidth=1)

    b1 = ax.boxplot(
        box1,
        positions=positions1,
        widths=0.3,#0.22
        patch_artist=True,
        boxprops=dict(facecolor="#4a4a4a", edgecolor="black", alpha=0.8, **common_props),
        medianprops=dict(color="white", linewidth=1.4),
        whiskerprops=dict(color="black", **common_props),
        capprops=dict(color="black", **common_props),
        flierprops=dict(marker='o', markersize=3, alpha=0.3)
    )

    #b3 = ax.boxplot(
    #    box3,
    #    positions=positions3,
    #    widths=0.22,
    #    patch_artist=True,
    #    boxprops=dict(facecolor="#9a9a9a", edgecolor="black", **common_props),
    #    medianprops=dict(color="black", linewidth=1.4),
    #    whiskerprops=dict(color="black", linestyle=":", **common_props),
    #    capprops=dict(color="black", linestyle=":", **common_props),
    #    flierprops=dict(marker='o', markersize=3, alpha=0.3)
    #)

    b2 = ax.boxplot(
        box2,
        positions=positions2,
        widths=0.3,#0.22
        patch_artist=True,
        boxprops=dict(facecolor="#d9d9d9", edgecolor="black", linewidth=1.2),
        medianprops=dict(color="black", linewidth=1.4),
        whiskerprops=dict(color="black", linestyle="--", **common_props),
        capprops=dict(color="black", linestyle="--", **common_props),
        flierprops=dict(marker='o', markersize=3, alpha=0.3)
    )

    ax.set_xticks(range(len(dims)))
    ax.set_xticklabels(dims)
    ax.grid(axis='y', linestyle='--', linewidth=0.5, alpha=0.5)
    ax.set_yscale("log")

    row = ax_idx // 2
    col = ax_idx % 2

    # X label only on bottom row
    if row == 1:
        ax.set_xlabel("Dimension")
    else:
        ax.set_title(titles[DIR])
        ax.set_xticklabels([])

    # Y label only on top-left plot (simplest shared look)
    if ax_idx in [0, 2]:
        ax.set_ylabel("rel. error")
    if ax_idx in [1, 3]:
        ax.set_ylabel("WF-SMC" if ax_idx == 1 else "SMC")
        ax.yaxis.set_label_position("right")
        ax.yaxis.tick_right()
    
    # Legend only on first row, second plot
    if ax_idx == 1:
        leg = ax.legend(
            [b1["boxes"][0],  b2["boxes"][0]],#b3["boxes"][0],
            [r"$\hat{Z}_T^{\mathrm{med}}$",  r"$\hat{Z}_T$"], #r"$\hat{Z}_T^{\mathrm{med},\prime}$",
            loc="upper right",
            frameon=True,
            facecolor="white",
            edgecolor="0.3",
            framealpha=0.7,
            fancybox=False,
            borderpad=0.6
        )
        leg.set_zorder(10)
        leg.get_frame().set_linewidth(0.8)

plt.tight_layout()
plt.savefig(DIRS[0] + "rel_error_plots.pdf", dpi=300)
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(1, len(DIRS), figsize=(5 * len(DIRS), 5))

if len(DIRS) == 1:
    axes = [axes]


for ax_idx, (DIR, ax) in enumerate(zip(DIRS, axes)):
    dims = list(data[DIR].keys())
    dims.sort()
    dims = list(filter(lambda dim: dim != 100, dims))

    box1 = [np.log(data[DIR][dim]["Zmed"][:200] / data[DIR][dim]["true_Z"]) for dim in dims]
    box2 = [np.log(data2[DIR][dim]["Z"][:200]   / data[DIR][dim]["true_Z"]) for dim in dims]

    positions1 = [i - 0.18 for i in range(len(dims))]
    positions2 = [i + 0.18 for i in range(len(dims))]

    common_props = dict(linewidth=1)

    b1 = ax.boxplot(
        box1,
        positions=positions1,
        widths=0.3,
        patch_artist=True,
        boxprops=dict(facecolor="#4a4a4a", edgecolor="black", alpha=0.8, **common_props),
        medianprops=dict(color="white", linewidth=1.4),
        whiskerprops=dict(color="black", **common_props),
        capprops=dict(color="black", **common_props),
        flierprops=dict(marker='o', markersize=3, alpha=0.3)
    )

    b2 = ax.boxplot(
        box2,
        positions=positions2,
        widths=0.3,
        patch_artist=True,
        boxprops=dict(facecolor="#d9d9d9", edgecolor="black", linewidth=1.2),
        medianprops=dict(color="black", linewidth=1.4),
        whiskerprops=dict(color="black", linestyle="--", **common_props),
        capprops=dict(color="black", linestyle="--", **common_props),
        flierprops=dict(marker='o', markersize=3, alpha=0.3)
    )

    ax.set_xticks(range(len(dims)))
    ax.set_xticklabels(dims)
    ax.grid(axis='y', linestyle='--', linewidth=0.5, alpha=0.5)
    ax.set_title(titles[DIR])

    if ax_idx == 0:
        ax.set_xlabel("Dimension")
        ax.set_ylabel("Log error")

        leg = ax.legend(
            [b1["boxes"][0], b2["boxes"][0]],
            [r"$\hat{Z}_T^{\mathrm{med}}$", r"$\hat{Z}_T$"],
            loc="lower left",
            frameon=True,
            facecolor="white",
            edgecolor="0.3",
            framealpha=0.7,
            fancybox=False,
            borderpad=0.6
        )
        leg.set_zorder(10)
        leg.get_frame().set_linewidth(0.8)

    else:
        # Put y-axis on the right for second plot
        ax.yaxis.tick_right()
        ax.yaxis.set_label_position("right")

plt.tight_layout()
plt.savefig(DIRS[0] + "log_errors_plots.pdf", dpi=300)
plt.show()